# ViShell — pipeline trên Colab hoặc Kaggle

Notebook mỏng: (mount Drive nếu là Colab) → cài đặt → chọn config → chạy
`python -m vishell pipeline` → hiện `REPORT.md`. Toàn bộ logic nằm trong package
`vishell/` — sửa code ở đó, không sửa trong notebook này (AGENT.md mục 9).

- **Colab**: Runtime > Change runtime type > GPU (T4). Output lưu lên Google Drive.
- **Kaggle**: Settings > Accelerator > GPU T4, và bật **Internet: On** (cần để `git clone`/`pip`).
  Output lưu ở `/kaggle/working` (chỉ giữ lại khi bấm *Save Version*).

In [ ]:
import os

if os.path.exists('/content'):  # Colab
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/vishell'
    WORK = '/content'
elif os.path.exists('/kaggle/working'):  # Kaggle
    ROOT = '/kaggle/working/vishell'
    WORK = '/kaggle/working'
else:
    raise RuntimeError('Không nhận ra Colab hay Kaggle — chạy trên laptop thì dùng `python -m vishell ...` trực tiếp.')
print('ROOT =', ROOT)

In [ ]:
REPO_URL = 'https://github.com/YOUR_ORG/TheWicknessHero.git'  # sửa lại URL thật của bạn
import subprocess
REPO_DIR = f'{WORK}/TheWicknessHero'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}/vishell
!pip install -q -e .[train,data]

In [ ]:
SMOKE = True  # True: chạy pipeline trên cấu hình smoke trước để bắt lỗi sớm
CONFIG = 'configs/smoke.yaml' if SMOKE else 'configs/colab_t4.yaml'  # đổi sang colab_a100.yaml nếu có A100/L4
print('Config:', CONFIG)

In [ ]:
# smoke.yaml tắt bước GPU cho laptop (train.enabled=false) — ở đây có GPU nên bật lại
!python -m vishell pipeline --config {CONFIG} --set paths.root={ROOT!r} --set train.enabled=true

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('REPORT.md', encoding='utf-8').read()))